# Induction Head Ablation Study

Run ablation experiments on candidate induction heads identified in notebook 05.

Tests whether heads that are nearby known GPT-2 induction heads in embedding space
also behave like induction heads when ablated.

## Core Metric:
**Loss on repeated sequences** - ablating an induction head should increase loss on `[A B C][A B C]...` sequences

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import numpy as np
import torch
from jaxtyping import Int
from torch import Tensor
from tqdm import tqdm

# attention-motifs ablation
from attention_motifs.ablation import (
    AblationMethod,
    HeadAblator,
    CandidateHeads,
    find_candidate_induction_heads,
    get_control_heads,
    generate_repeated_sequences,
    repeated_sequence_loss,
    icl_score,
    run_ablation_experiment,
)
from attention_motifs.ablation.experiment import (
    ExperimentConfig,
    ExperimentResults,
    analyze_results,
    identify_induction_heads,
)
from attention_motifs.ablation.candidates import get_known_induction_heads
from attention_motifs.ablation.ablate import heads_from_strings
from attention_motifs.ablation.data import RepeatedSequence
from attention_motifs.features.analysis import DistanceTensorResult

from transformer_lens import HookedTransformer

In [2]:
# config
pl.Config.set_tbl_rows(30)
PATH_BASE: Path = Path("../data/")
DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


# Load candidates from notebook 05

In [ ]:
# load candidates (or regenerate if needed)
candidates_path: Path = PATH_BASE / "ablation" / "candidates.json"

if candidates_path.exists():
    CANDIDATES: CandidateHeads = CandidateHeads.read(candidates_path)
    print(f"Loaded {len(CANDIDATES.candidates_by_model)} models worth of candidates")
else:
    print("Candidates not found, regenerating...")
    HEAD_DISTS: DistanceTensorResult = DistanceTensorResult.read_raw(PATH_BASE / "features" / "head_dists_raw")
    CANDIDATES = find_candidate_induction_heads(HEAD_DISTS, k_neighbors=20)

In [4]:
# show available models
for model, candidates in CANDIDATES.candidates_by_model.items():
    print(f"{model}: {len(candidates)} candidates")

gpt2-medium: 37 candidates
gemma-2-2b: 7 candidates
Llama-3-2-1B: 33 candidates
pythia-1b: 12 candidates
gemma-2b: 3 candidates


# Sanity check: Verify ablation works on known GPT-2 induction heads

In [5]:
# load gpt2-small and test known induction heads
model_gpt2: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small", device=DEVICE)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer


In [6]:
# generate test sequences
sequences: list[RepeatedSequence] = generate_repeated_sequences(
    tokenizer=model_gpt2.tokenizer,
    n_sequences=50,
    seq_length=25,
    n_repetitions=4,
    seed=42,
    device=DEVICE,
)
print(f"Generated {len(sequences)} test sequences")

Generated 50 test sequences


In [7]:
# baseline loss (no ablation)
baseline_loss: float = repeated_sequence_loss(model_gpt2, sequences)
print(f"Baseline loss on repeated sequences: {baseline_loss:.4f}")

Baseline loss on repeated sequences: 0.2510


In [ ]:
# test ablating known induction heads (from AttentionPedia)
known_head_strs: list[str] = get_known_induction_heads()
KNOWN_INDUCTION_HEADS: list[tuple[int, int]] = heads_from_strings(known_head_strs)
print(f"Known induction heads from AttentionPedia: {known_head_strs}")

ablator: HeadAblator = HeadAblator(model_gpt2)

# compute mean activations for mean ablation
calibration_tokens: list[Int[Tensor, "seq_len"]] = [s.tokens for s in sequences[:30]]
ablator.compute_mean_activations(calibration_tokens)

# store results for known induction heads
known_results: list[dict] = []

for method in [AblationMethod.ZERO, AblationMethod.MEAN]:
    print(f"\n{method.name} Ablation Results:")
    print("-" * 50)
    for layer, head in KNOWN_INDUCTION_HEADS:
        with ablator.ablate_heads([(layer, head)], method):
            ablated_loss = repeated_sequence_loss(model_gpt2, sequences)
        loss_increase = ablated_loss - baseline_loss
        print(f"L{layer}:H{head}: loss={ablated_loss:.4f} (increase={loss_increase:.4f})")
        known_results.append({
            "layer": layer,
            "head": head,
            "method": method.name,
            "loss": ablated_loss,
            "loss_increase": loss_increase,
            "is_known_induction": True,
        })

In [ ]:
# test ALL non-induction heads
KNOWN_SET: set[tuple[int, int]] = set(KNOWN_INDUCTION_HEADS)
all_results: list[dict] = []

n_layers = model_gpt2.cfg.n_layers
n_heads = model_gpt2.cfg.n_heads
total_heads = n_layers * n_heads - len(KNOWN_SET)

print(f"Testing {total_heads} non-induction heads (ZERO ablation)...")

for layer in tqdm(range(n_layers), desc="Layers"):
    for head in range(n_heads):
        if (layer, head) in KNOWN_SET:
            continue
        with ablator.ablate_heads([(layer, head)], AblationMethod.ZERO):
            ablated_loss = repeated_sequence_loss(model_gpt2, sequences)
        all_results.append({
            "layer": layer,
            "head": head,
            "method": "ZERO",
            "loss": ablated_loss,
            "loss_increase": ablated_loss - baseline_loss,
            "is_known_induction": False,
        })

non_induction_df = pl.DataFrame(all_results)
print(f"\nTested {len(all_results)} heads")

In [ ]:
# histogram: distribution of loss increase for all heads
fig, ax = plt.subplots(figsize=(12, 6))

# get known induction head results (ZERO method only)
known_zero = [r for r in known_results if r["method"] == "ZERO"]
known_losses = [r["loss_increase"] for r in known_zero]
non_induction_losses = non_induction_df["loss_increase"].to_numpy()

# histogram of non-induction heads
ax.hist(non_induction_losses, bins=30, alpha=0.7, color="steelblue", 
        label=f"Non-induction heads (n={len(non_induction_losses)})", edgecolor="black")

# mark known induction heads with vertical lines
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(known_zero)))
for i, r in enumerate(known_zero):
    ax.axvline(r["loss_increase"], color=colors[i], linestyle="--", linewidth=2,
               label=f"L{r['layer']}:H{r['head']} ({r['loss_increase']:.2f})")

ax.set_xlabel("Loss Increase (ablated - baseline)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of Loss Increase when Ablating Each Head (ZERO ablation)", fontsize=14)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.axvline(0, color="gray", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.savefig("figures/ablation_histogram_gpt2_small.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# combine all results into a single dataframe and sort by loss increase
all_heads_df = pl.concat([
    non_induction_df,
    pl.DataFrame([r for r in known_results if r["method"] == "ZERO"]),
])

# sort by loss increase descending
all_heads_sorted = all_heads_df.sort("loss_increase", descending=True)

# show top 30 heads
print("Top 30 heads by loss increase (ZERO ablation):")
print("=" * 60)
all_heads_sorted.head(30)

In [ ]:
# summary statistics
print("Summary Statistics:")
print("=" * 60)

known_df = all_heads_sorted.filter(pl.col("is_known_induction"))
unknown_df = all_heads_sorted.filter(~pl.col("is_known_induction"))

print(f"\nKnown induction heads ({len(known_df)}):")
print(f"  Mean loss increase: {known_df['loss_increase'].mean():.4f}")
print(f"  Min: {known_df['loss_increase'].min():.4f}, Max: {known_df['loss_increase'].max():.4f}")

print(f"\nNon-induction heads ({len(unknown_df)}):")
print(f"  Mean loss increase: {unknown_df['loss_increase'].mean():.4f}")
print(f"  Min: {unknown_df['loss_increase'].min():.4f}, Max: {unknown_df['loss_increase'].max():.4f}")

# how many non-induction heads have higher loss increase than the median known head?
median_known = known_df["loss_increase"].median()
higher_than_median = unknown_df.filter(pl.col("loss_increase") > median_known)
print(f"\nNon-induction heads with loss increase > median known ({median_known:.4f}): {len(higher_than_median)}")

# show those heads
if len(higher_than_median) > 0:
    print("\nThese 'surprising' high-impact non-induction heads:")
    print(higher_than_median.select(["layer", "head", "loss_increase"]).sort("loss_increase", descending=True))

# Run full experiment on candidate model

In [ ]:
# select model to test (change this to test different models)
TARGET_MODEL: str = "pythia-1b"  # or "gemma-2b", etc.

# get candidates and controls
model_candidates: list[tuple[str, float]] = CANDIDATES.get_top_candidates(TARGET_MODEL, n=10)
print(f"Top 10 candidates for {TARGET_MODEL}:")
for head, score in model_candidates:
    print(f"  {head}: {score:.3f}")

In [ ]:
# run experiment
config: ExperimentConfig = ExperimentConfig(
    n_sequences=100,
    seq_length=25,
    n_repetitions=4,
    ablation_methods=[AblationMethod.ZERO, AblationMethod.MEAN],
    seed=42,
)

candidate_heads: list[str] = [h for h, _ in model_candidates]

# get control heads from distance matrix
HEAD_DISTS: DistanceTensorResult = DistanceTensorResult.read_raw(PATH_BASE / "features" / "head_dists_raw")
control_heads: list[str] = get_control_heads(HEAD_DISTS, CANDIDATES, TARGET_MODEL, n_controls=5, method="far")

print(f"\nControl heads: {control_heads}")

In [ ]:
# run the experiment
results: ExperimentResults = run_ablation_experiment(
    model_name=TARGET_MODEL,
    candidate_heads=candidate_heads,
    control_heads=control_heads,
    config=config,
    device=DEVICE,
)

In [ ]:
# view results
results_df: pl.DataFrame = results.to_dataframe()
results_df.sort("loss_increase", descending=True)

# Analyze and visualize results

In [ ]:
# summary statistics
summary: pl.DataFrame = analyze_results(results)
summary

In [ ]:
# plot loss increase for candidates vs controls
fig, ax = plt.subplots(figsize=(8, 5))

# separate candidates and controls
candidate_set: set[str] = set(candidate_heads)
control_set: set[str] = set(control_heads)

df_zero: pl.DataFrame = results_df.filter(pl.col("ablation_method") == "zero")

# mark as candidate or control
df_zero = df_zero.with_columns(
    pl.when(pl.col("head").is_in(candidate_set))
    .then(pl.lit("candidate"))
    .otherwise(pl.lit("control"))
    .alias("type")
)

# loss increase comparison
candidate_losses = df_zero.filter(pl.col("type") == "candidate")["loss_increase"].to_numpy()
control_losses = df_zero.filter(pl.col("type") == "control")["loss_increase"].to_numpy()

ax.boxplot([candidate_losses, control_losses], labels=["Candidates", "Controls"])
ax.set_ylabel("Loss Increase")
ax.set_title("Loss Increase: Candidates vs Controls (ZERO ablation)")
ax.axhline(y=0, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
Path("figures").mkdir(exist_ok=True)
plt.savefig(f"figures/ablation_results_{TARGET_MODEL.replace('/', '_')}.pdf", bbox_inches="tight")

In [ ]:
# identify which candidates are likely induction heads (by loss increase only)
induction_heads: list[str] = identify_induction_heads(
    results,
    loss_threshold=0.3,
    prefix_threshold=0.0,  # ignore prefix score for now
)

print(f"\nHeads classified as likely induction heads ({len(induction_heads)}):")
for head in induction_heads:
    print(f"  {head}")

# Save results

In [ ]:
# save results
output_dir: Path = PATH_BASE / "ablation"
output_dir.mkdir(parents=True, exist_ok=True)

results.save(output_dir / f"{TARGET_MODEL.replace('/', '_')}_results.json")
results_df.write_csv(output_dir / f"{TARGET_MODEL.replace('/', '_')}_results.csv")

print(f"Results saved to {output_dir}")

# Compare zero vs mean ablation

In [ ]:
# compare ablation methods
fig, ax = plt.subplots(figsize=(8, 6))

zero_losses = results_df.filter(pl.col("ablation_method") == "zero")["loss_increase"].to_numpy()
mean_losses = results_df.filter(pl.col("ablation_method") == "mean")["loss_increase"].to_numpy()

ax.scatter(zero_losses, mean_losses, alpha=0.7)
ax.plot([min(zero_losses), max(zero_losses)], [min(zero_losses), max(zero_losses)], 'r--', label='y=x')
ax.set_xlabel("Zero Ablation Loss Increase")
ax.set_ylabel("Mean Ablation Loss Increase")
ax.set_title("Zero vs Mean Ablation")
ax.legend()

plt.tight_layout()
Path("figures").mkdir(exist_ok=True)
plt.savefig(f"figures/ablation_methods_{TARGET_MODEL.replace('/', '_')}.pdf", bbox_inches="tight")